# Day 22 Capstone — Energy-Demand Forecasting with LSTM + Attention

**Objective:** Build an end-to-end, multi-horizon hourly energy-demand forecaster using a fully synthetic series. Compare a seasonal-naive benchmark, an LSTM encoder–decoder, and an LSTM encoder–decoder with attention.

**Forecast setup**
- Look-back: **168 hours** (one full week)
- Forecast horizon: **24 hours**
- Chronological split: **70% train / 15% validation / 15% test**
- Metrics: **MAE, RMSE, MAPE**
- No external data, URLs, or shuffled time-series splits.


## 1. Imports, reproducibility, and synthetic data

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RNG = 42
np.random.seed(RNG)
tf.random.set_seed(RNG)

print("TensorFlow:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Generate 2 years of hourly demand, entirely offline
N = 24 * 7 * 104  # 104 weeks
t = np.arange(N)

trend = 0.0015 * t
daily = 8.0 * np.sin(2 * np.pi * t / 24 - 1.0)
weekly = 4.0 * np.sin(2 * np.pi * t / (24 * 7))
weekend = -3.0 * (((t // 24) % 7) >= 5)
noise = np.random.normal(0, 1.2, N)

demand = (50.0 + trend + daily + weekly + weekend + noise).astype("float32")

print("series length:", demand.shape,
      " mean:", round(float(demand.mean()), 2),
      " min:", round(float(demand.min()), 2),
      " max:", round(float(demand.max()), 2))

plt.figure(figsize=(15, 4))
plt.plot(demand[:24*14], linewidth=1)
plt.title("First two weeks of synthetic hourly demand")
plt.xlabel("Hour")
plt.ylabel("Demand")
plt.grid(alpha=0.25)
plt.show()

## 2. Frame the forecasting task

Each example contains the previous **168 hourly observations** and predicts the next **24 hours**. The split is based on the timestamp of each target window, so validation and test targets occur strictly after training targets. The input window may include earlier observations immediately before its target; this is realistic forecasting context, not future leakage.

In [ ]:
LOOK_BACK = 168
HORIZON = 24

# Chronological boundaries on the original series
train_end = int(N * 0.70)
val_end = int(N * 0.85)

# A window is assigned to a split according to the end of its target.
X_all, y_all, target_starts = [], [], []
for start in range(N - LOOK_BACK - HORIZON + 1):
    target_start = start + LOOK_BACK
    target_end = target_start + HORIZON
    X_all.append(demand[start:target_start])
    y_all.append(demand[target_start:target_end])
    target_starts.append(target_start)

X_all = np.asarray(X_all, dtype="float32")[..., None]
y_all = np.asarray(y_all, dtype="float32")
target_starts = np.asarray(target_starts)

train_mask = target_starts + HORIZON <= train_end
val_mask = (target_starts >= train_end) & (target_starts + HORIZON <= val_end)
test_mask = target_starts >= val_end

X_train_raw, y_train_raw = X_all[train_mask], y_all[train_mask]
X_val_raw, y_val_raw = X_all[val_mask], y_all[val_mask]
X_test_raw, y_test_raw = X_all[test_mask], y_all[test_mask]

print("Train:", X_train_raw.shape, y_train_raw.shape)
print("Validation:", X_val_raw.shape, y_val_raw.shape)
print("Test:", X_test_raw.shape, y_test_raw.shape)

assert len(X_train_raw) > 0 and len(X_val_raw) > 0 and len(X_test_raw) > 0
assert target_starts[train_mask].max() + HORIZON <= train_end
assert target_starts[val_mask].min() >= train_end
assert target_starts[test_mask].min() >= val_end

## 3. Normalize using training data only

In [ ]:
# Fit one scalar mean/std on the raw observations available to training.
# No validation or test values contribute to these statistics.
scaler_mean = float(demand[:train_end].mean())
scaler_std = float(demand[:train_end].std())
if scaler_std == 0:
    raise ValueError("Training standard deviation is zero.")

def scale(a):
    return ((a - scaler_mean) / scaler_std).astype("float32")

def inverse_scale(a):
    return a * scaler_std + scaler_mean

X_train = scale(X_train_raw)
X_val = scale(X_val_raw)
X_test = scale(X_test_raw)

y_train = scale(y_train_raw)
y_val = scale(y_val_raw)
y_test = scale(y_test_raw)

print(f"Training scaler mean={scaler_mean:.3f}, std={scaler_std:.3f}")
print("Scaled train input mean/std:", round(float(X_train.mean()), 3),
      round(float(X_train.std()), 3))

## 4. Seasonal-naive baseline

The baseline predicts each hour using the value from **168 hours earlier** (the same hour on the previous week). Because the input spans exactly one week, these values are the first 24 points of each look-back window. This is a meaningful benchmark for a series with daily and weekly seasonal structure.

In [ ]:
# X_raw shape: (samples, look_back, 1)
# For target hours t+1 ... t+24, the corresponding weekly-lag values
# are the first 24 hours in the preceding 168-hour input window.
baseline_pred = X_test_raw[:, :HORIZON, 0]
baseline_pred.shape, y_test_raw.shape

## 5. Build and train the LSTM encoder–decoder

In [ ]:
def build_lstm_model():
    encoder_inputs = keras.Input(shape=(LOOK_BACK, 1), name="history")
    _, state_h, state_c = layers.LSTM(
        48, return_state=True, name="encoder"
    )(encoder_inputs)

    decoder_seed = layers.RepeatVector(HORIZON, name="repeat_context")(state_h)
    decoder_outputs = layers.LSTM(
        48, return_sequences=True, name="decoder"
    )(decoder_seed, initial_state=[state_h, state_c])

    outputs = layers.Dense(1, name="forecast")(decoder_outputs)
    outputs = layers.Reshape((HORIZON,), name="horizon_output")(outputs)

    model = keras.Model(encoder_inputs, outputs, name="lstm_encoder_decoder")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

lstm_model = build_lstm_model()
lstm_model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
    )
]

history_lstm = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

plt.figure(figsize=(8, 4))
plt.plot(history_lstm.history["loss"], label="train")
plt.plot(history_lstm.history["val_loss"], label="validation")
plt.title("LSTM training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss (scaled)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Add attention over encoder states

The attention model keeps the full sequence of encoder outputs. The decoder produces one hidden state per forecast step; Keras `Attention` uses each decoder state as a query and the encoder sequence as keys/values. The resulting context is concatenated with the decoder output before the forecast layer. This lets each forecast step select relevant parts of the historical week.

In [ ]:
def build_attention_model():
    encoder_inputs = keras.Input(shape=(LOOK_BACK, 1), name="history")
    encoder_outputs, state_h, state_c = layers.LSTM(
        48, return_sequences=True, return_state=True, name="encoder"
    )(encoder_inputs)

    decoder_seed = layers.RepeatVector(HORIZON, name="repeat_context")(state_h)
    decoder_outputs = layers.LSTM(
        48, return_sequences=True, name="decoder"
    )(decoder_seed, initial_state=[state_h, state_c])

    context = layers.Attention(name="temporal_attention")(
        [decoder_outputs, encoder_outputs]
    )
    combined = layers.Concatenate(name="decoder_plus_context")(
        [decoder_outputs, context]
    )
    outputs = layers.Dense(1, name="forecast")(combined)
    outputs = layers.Reshape((HORIZON,), name="horizon_output")(outputs)

    model = keras.Model(encoder_inputs, outputs, name="lstm_with_attention")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

attention_model = build_attention_model()
attention_model.summary()

In [ ]:
# Reset the seed so the second experiment is reproducible.
tf.keras.utils.set_random_seed(RNG)

history_attention = attention_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
        )
    ],
    verbose=1
)

plt.figure(figsize=(8, 4))
plt.plot(history_attention.history["loss"], label="train")
plt.plot(history_attention.history["val_loss"], label="validation")
plt.title("LSTM + attention training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss (scaled)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 7. Evaluate on the untouched test set

In [ ]:
def calculate_metrics(actual, predicted):
    actual = np.asarray(actual).reshape(-1)
    predicted = np.asarray(predicted).reshape(-1)
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    # Demand is strictly positive in this synthetic dataset.
    mape = np.mean(np.abs((actual - predicted) / np.maximum(np.abs(actual), 1e-8))) * 100
    return {"MAE": mae, "RMSE": rmse, "MAPE (%)": mape}

lstm_pred_scaled = lstm_model.predict(X_test, verbose=0)
attention_pred_scaled = attention_model.predict(X_test, verbose=0)

lstm_pred = inverse_scale(lstm_pred_scaled)
attention_pred = inverse_scale(attention_pred_scaled)

results = pd.DataFrame(
    {
        "Seasonal naive": calculate_metrics(y_test_raw, baseline_pred),
        "LSTM encoder-decoder": calculate_metrics(y_test_raw, lstm_pred),
        "LSTM + attention": calculate_metrics(y_test_raw, attention_pred),
    }
).T

display(results.round(3))
print("Best model by test MAE:", results["MAE"].idxmin())

In [ ]:
# Plot one representative test forecast
example = min(10, len(y_test_raw) - 1)
hours = np.arange(HORIZON)

plt.figure(figsize=(12, 4))
plt.plot(hours, y_test_raw[example], marker="o", label="Actual")
plt.plot(hours, baseline_pred[example], marker=".", label="Seasonal naive")
plt.plot(hours, lstm_pred[example], marker=".", label="LSTM")
plt.plot(hours, attention_pred[example], marker=".", label="LSTM + attention")
plt.title("24-hour test forecast example")
plt.xlabel("Forecast hour")
plt.ylabel("Demand")
plt.xticks(hours[::2])
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### Interpretation checklist

- Compare every learned model with the weekly seasonal-naive benchmark; a neural network is not automatically better.
- **MAE** is the average absolute error in demand units. **RMSE** penalizes larger misses more strongly. **MAPE** expresses average absolute error as a percentage of actual demand.
- If attention does not improve the test metrics, report that honestly. The synthetic process is dominated by stable daily/weekly patterns, so a simple seasonal baseline can be difficult to beat.
- Test metrics are a final one-time estimate. Do not tune architecture or epochs based on test performance.

## 8. Written reflection (150–250 words)

This experiment framed hourly demand forecasting as a supervised multi-output learning problem. Each example used the previous 168 hours to predict the next 24, preserving chronological order throughout the train, validation, and test split. Normalization statistics were calculated only from the training period, preventing future information from leaking into preprocessing. A weekly seasonal-naive forecast was used as the benchmark because it reuses demand from the same hours of the previous week.

The LSTM encoder–decoder learned a compact representation of the preceding week and decoded it into a 24-hour forecast. The attention version retained all encoder time steps and let each decoder step weight historical states differently. The metric table should be interpreted against the baseline, not in isolation: MAE measures typical absolute error, RMSE gives greater weight to large errors, and MAPE expresses error relative to demand. Attention earns its additional complexity only if it produces a meaningful and consistent improvement on held-out data.

The decoder here is not trained with teacher forcing: it receives a repeated encoder context rather than previous ground-truth target values. Consequently, inference uses the same inputs as training and does not accumulate autoregressive feedback errors. A limitation is that the synthetic series has stable, known seasonal structure and cannot represent all real operational shocks. The strongest defensible conclusion is therefore based on the observed test metrics, with model complexity justified only by measurable gains.

---

**Reproducibility note:** The data are generated in this notebook with a fixed random seed. Run all cells from top to bottom. Training is intentionally small and CPU-compatible; exact metrics can vary slightly with TensorFlow version and hardware.